In [1]:
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [3]:
RUTA_CENSO_ZONAL = "../Parquet/Cartografía_censo2024_R13_Zonal.parquet"
RUTA_CENSO_COMUNAL = "../Parquet/Cartografía_censo2024_R13_Comunal.parquet"
SEED = 123

In [4]:
censo_zonal = gpd.read_parquet(RUTA_CENSO_ZONAL).rename_geometry("geometry")

es_gran_santiago = (censo_zonal["PROVINCIA"] == "SANTIAGO") | (
    censo_zonal["COMUNA"].isin(["PUENTE ALTO", "SAN BERNARDO"])
)
censo_zonal = censo_zonal[es_gran_santiago].reset_index(drop=True)

censo_comunal = gpd.read_parquet(RUTA_CENSO_COMUNAL).rename_geometry("geometry")
censo_comunal = censo_comunal[
    (censo_comunal["PROVINCIA"] == "SANTIAGO") | (censo_comunal["COMUNA"].isin(["PUENTE ALTO", "SAN BERNARDO"]))
].reset_index(drop=True)

print(f"Zonas censales en Gran Santiago: {len(censo_zonal)}")
print(f"Comunas: {len(censo_comunal)}")

Zonas censales en Gran Santiago: 1642
Comunas: 34


# CONTRUCCIÓN DE INDICADORES
Generar tazas en base a las variables que trae el parquet del CENSO

In [7]:
# Filro para excluir xonas censales con 0 personas y viviendas.
usable = (censo_zonal['n_per']>0) & (censo_zonal['n_vp_ocupada']>0)

censo_zonal = censo_zonal[usable].reset_index(drop=True)

In [11]:
# Se generan los indicadores
censo_zonal['pct_educ_sup'] = censo_zonal['n_cine_terciaria_maestria_doctorado']/censo_zonal['n_per'] * 100
censo_zonal["pct_hacinamiento"] = censo_zonal["n_viv_hacinadas"] / censo_zonal["n_vp_ocupada"] * 100
censo_zonal["pct_dificultad_ver"] = censo_zonal["n_dificultad_ver"] / censo_zonal["n_per"] * 100

indicadores = ["pct_educ_sup", "pct_hacinamiento", "pct_dificultad_ver"]
censo_zonal[indicadores].describe().round(1)

,pct_educ_sup,pct_hacinamiento,pct_dificultad_ver
count,1640.0,1640.0,1640.0
mean,28.4,6.2,4.0
std,17.7,4.3,1.7
min,0.0,0.0,0.0
25%,13.5,2.8,2.6
50%,23.4,5.7,4.0
75%,41.5,8.8,5.2
max,73.5,50.0,10.7


In [12]:
# Correlación

censo_zonal["pct_dificultad_ver"].corr(censo_zonal["pct_educ_sup"])

-0.7997723200592695